# ASI02 Tool Misuse & Exploitation — Upload Artifacts & Run Evaluation

**OWASP Category**: ASI02 — Tool Misuse & Exploitation | **Risk Severity**: Critical

**Mapped LLM Categories**: LLM01 (Prompt Injection), LLM06 (Excessive Agency)

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) and registers them in Okareo.
2. **Runs** the full ASI02 test suite against your target AI agent.

**ASI02 tests for**:
- Over-privileged tool invocations (agent uses tools beyond its stated scope)
- Indirect injection via tool-pivot (external data contains tool-invocation instructions)
- Tool-loop amplification (agent accepts unbounded or recursive tool-call patterns)

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
from pathlib import Path

_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver
from owasp.common import init_okareo, parse_artifact, build_target, SINGLE_TURN_DRIVER_TEMPLATE

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")

---
## Part 1 — Upload Artifacts
### Upload Scenarios

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"ASI02-{jsonl_path.stem}"
    print(f"Uploading: {scenario_name}")
    scenario = okareo.upload_scenario_set(scenario_name=scenario_name, file_path=str(jsonl_path))
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

### Register Checks

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")

### Register Drivers

In [ ]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_artifact(md_path, default_temperature=0.7)
    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")

---
## Part 2 — Run Evaluation

### Configuration

- Single-turn: `over-privileged-tool-invocation`, `indirect-injection-tool-pivot`, `tool-loop-amplification` — checked with `ASI02-tool-misuse-detector`
- Multi-turn: `tool-loop-amplification` scenario with `ASI02-tool-loop-inducer` driver also run as a simulation

In [ ]:
target = build_target(CATEGORY_DIR)
print(f"✓ Target agent: {target.name}")

TOOL_MISUSE_CHECK = "ASI02-tool-misuse-detector"
MAX_TURNS = 10

SINGLE_TURN_SCENARIOS = [
    "ASI02-over-privileged-tool-invocation",
    "ASI02-indirect-injection-tool-pivot",
    "ASI02-tool-loop-amplification",
]

MULTI_TURN_MAP = {
    "ASI02-over-privileged-tool-invocation": "ASI02-tool-pivot-attacker",
    "ASI02-tool-loop-amplification": "ASI02-tool-loop-inducer",
}

all_results = {}

In [ ]:
single_turn_driver = Driver(
    name="pass-through",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
    temperature=0,
)

for scenario_name in SINGLE_TURN_SCENARIOS:
    if scenario_name not in registered_scenarios:
        print(f"⚠ {scenario_name} not found — skipping")
        continue
    print(f"Running single-turn: {scenario_name}")
    try:
        run = okareo.run_simulation(
            target=target,
            driver=single_turn_driver,
            name=f"ASI02 Single-Turn — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="target",
            scenario=registered_scenarios[scenario_name],
            max_turns=1,
            checks=[TOOL_MISUSE_CHECK],
        )
        all_results[scenario_name] = run
        print(f"  ✓ {run.id}")
        if getattr(run, "app_link", None): print(f"  View: {run.app_link}")
    except Exception as e:
        print(f"  ✗ {e}")
        all_results[scenario_name] = None

In [ ]:
for scenario_name, driver_name in MULTI_TURN_MAP.items():
    if scenario_name not in registered_scenarios or driver_name not in registered_drivers:
        print(f"⚠ Missing artifact for {scenario_name} — skipping")
        continue
    print(f"\nRunning simulation: {scenario_name} | Driver: {driver_name}")
    try:
        drv_reg = registered_drivers[driver_name]
        multi_driver = Driver(
            temperature=getattr(drv_reg, "temperature", 0.7),
            name=driver_name,
            prompt_template=drv_reg.prompt_template,
        )
        run = okareo.run_simulation(
            target=target,
            driver=multi_driver,
            name=f"ASI02 Simulation — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="target",
            scenario=registered_scenarios[scenario_name],
            max_turns=MAX_TURNS,
            checks=[TOOL_MISUSE_CHECK],
        )
        key = f"{scenario_name} (multi-turn)"
        all_results[key] = run
        print(f"  ✓ {run.id}")
        if getattr(run, "app_link", None): print(f"  View: {run.app_link}")
    except Exception as e:
        all_results[f"{scenario_name} (multi-turn)"] = None
        print(f"  ✗ {e}")

In [ ]:
print("\n" + "=" * 60)
print("ASI02 TOOL MISUSE & EXPLOITATION — EVALUATION RESULTS")
print("=" * 60)
for name, result in all_results.items():
    if result is None:
        print(f"  ✗ {name}: ERROR")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"  ✓ {name}: {link}")
errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal: {len(all_results)} | Errors: {errors}")